# ProToken MLSys (artifact) — Chameleon Cloud reproduction

This notebook reproduces the ProToken MLSys paper results on Chameleon Cloud using the `chi` Python client.

## What you’ll do

- Provision a GPU node from an existing Chameleon lease
- Install dependencies on the node
- Run `reproduce.sh` for **RQ1–RQ4** (SmolLM + `coding` dataset)
- Pull generated figures back to this notebook and compare against the included reference outputs

## Requirements

- CHI-enabled Jupyter environment (logged in / configured)
- An existing lease with **≥ NVIDIA A100** GPU
  - The PyTorch version used here does **not** support older GPU architectures
- 512G usable RAM
- 512G of available space in `/tmp`

## Conventions

- Run cells **top to bottom**
- When you see `<...>` placeholders, replace them before running the next cell

In [ ]:
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

## Select your lease

Replace `<name-of-your-lease>` with the name of an existing CHI lease that includes suitable GPU nodes, then run the next cell to bind this notebook session to that lease.

- **Tip**: the lease must include at least one node reservation you can use for a server.

In [ ]:
lease_name = "<name-of-your-lease>"
l = lease.get_lease(lease_name)
l.show()

## Provision a GPU server

This creates a server on the reserved node using a CUDA-enabled image, then attaches a floating IP and verifies connectivity.

- **Expected outcome**: the connectivity check succeeds, and you can SSH to the node via the floating IP.

In [ ]:
username = os.getenv('USER')
s = server.Server(
    f"node-{lease_name}-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-CUDA"
)
s.submit(idempotent=True)

In [ ]:
s.associate_floating_ip()

In [ ]:
floating_ip = s.get_floating_ip()
print(f"Server floating IP: {floating_ip}")

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")

## Install and run on the node

The next few cells run commands on the provisioned node:

- Update packages
- Install `uv`
- Clone the `protoken` repository
- Install dependencies (with the `chameleon` extra)
- Run `reproduce.sh` for RQ1–RQ4 with:
  - **model**: `smollm`
  - **dataset**: `coding`

**Runtime note**: each RQ may take several minutes.

For each RQ, follow the instructions in the markdown cell above the command, then compare the resulting figures to the reference outputs included in this repo.

In [ ]:
s.execute("sudo apt update")

In [ ]:
s.execute("curl -LsSf https://astral.sh/uv/install.sh | sh")

In [ ]:
s.execute("git clone https://github.com/ahmayun/protoken.git")

In [ ]:
s.execute("cd protoken && ~/.local/bin/uv sync --extra chameleon")

## Generate a temporary SSH key

Run the next two cells to create a temporary keypair and print the public key.

In [ ]:
!ssh-keygen -t ed25519 -f ./temporary-key -N ""

In [ ]:
!cat temporary-key.pub

Copy the printed **public key** (the line ending in something like `... user@host`). You will paste it into the following cell where it says `<replace-with-your-public-key>`.

In [ ]:
s.execute("""echo "<replace-with-your-public-key>" >> ~/.ssh/authorized_keys""")

## RQ1 — Token-level provenance attribution accuracy (Fig. 2 & 3)

Run the next cell to generate RQ1 figures, then pull them back with the `scp` cell below.

### Reference figures (for the outputs ahead)

- **RQ1 (a) — Main accuracy** (reference: `reference-outputs/rq1-a.png`)

![RQ1(a) reference](reference-outputs/rq1-a.png)

- **RQ1 (b) — Client contribution distributions** (reference: `reference-outputs/rq1-b.png`)

![RQ1(b) reference](reference-outputs/rq1-b.png)

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --dataset coding --rq1")

In [ ]:
!mkdir rq1

In [ ]:
!scp -i ./temporary-key -o StrictHostKeyChecking=no cc@{floating_ip}:~/protoken/results/graphs/rq1/*.png ./rq1

In [ ]:
from IPython.display import Image

### RQ1(a) — Your output

Compare the plot below with the **RQ1(a) reference** shown above (the reference is provided for *this* output).

**What to look for**: the ProToken attribution accuracy curve should be higher than the baselines.

In [ ]:
Image("rq1/HuggingFaceTB_SmolLM2-360M-Instruct_rounds-10_epochs-1_clients6-per-round-6_Datasets-_'coding'_-None_partitioning-iid_Backdoor-True_Unsloth-False_Lora-False.png")


### RQ1(b) — Your output

Compare the plot below with the **RQ1(b) reference** shown above (the reference is provided for *this* output).

**What to look for**: responsible clients’ mean contribution accuracy (box plots) should be higher (closer to 1) than non-responsible clients’.

In [ ]:
Image("rq1/HuggingFaceTB_SmolLM2-360M-Instruct_rounds-10_epochs-1_clients6-per-round-6_Datasets-_'coding'_-None_partitioning-iid_Backdoor-True_Unsloth-False_Lora-False_boxplots.png")


## RQ2 — Effect of gradient weighting (relevance filtering) (Fig. 4)

Run the next cell to generate the RQ2 figure, then pull it back with the `scp` cell below.

### Reference figure (for the output ahead)

![RQ2 reference](reference-outputs/rq2.png)

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --dataset coding --rq2")

In [ ]:
!mkdir rq2

In [ ]:
!scp -i ./temporary-key -o StrictHostKeyChecking=no cc@{floating_ip}:~/protoken/results/graphs/rq2/*.png ./rq2

### RQ2 — Your output

Compare the plot below with the **RQ2 reference** shown above (the reference is provided for *this* output).

**What to look for**: the **"gradients enabled"** bar should be higher than **"gradients disabled"**.

In [ ]:
Image("rq2/gradient_ablation_bar_chart.png")

## RQ3 — Computational overhead vs. layer count (Fig. 5 — Tractability)

Run the next cell to generate the RQ3 figure, then pull it back with the `scp` cell below.

### Reference figure (for the output ahead)

![RQ3 reference](reference-outputs/rq3.png)

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --dataset coding --rq3")

In [ ]:
!mkdir rq3

In [ ]:
!scp -i ./temporary-key -o StrictHostKeyChecking=no cc@{floating_ip}:~/protoken/results/graphs/rq3/*.png ./rq3

### RQ3 — Your output

Compare the plot below with the **RQ3 reference** shown above (the reference is provided for *this* output).

**What to look for**: average provenance time should increase as the number of layers increases.

In [ ]:
Image("rq3/overhead-and-tool-accuracy.png")

## RQ4 — Scaling with more clients (Fig. 6 & 7)

Run the next cell to generate RQ4 figures, then pull them back with the `scp` cell below.

### Reference figures (for the outputs ahead)

- **RQ4 (a) — Scalability (accuracy)** (reference: `reference-outputs/rq4-a.png`)

![RQ4(a) reference](reference-outputs/rq4-a.png)

- **RQ4 (b) — Scalability (client contributions)** (reference: `reference-outputs/rq4-b.png`)

![RQ4(b) reference](reference-outputs/rq4-b.png)

In [ ]:
s.execute("cd protoken && PATH=$PATH:~/.local/bin ./reproduce.sh --model smollm --dataset coding --rq4")

In [ ]:
!mkdir rq4

In [ ]:
!scp -i ./temporary-key -o StrictHostKeyChecking=no cc@{floating_ip}:~/protoken/results/graphs/rq4/*.png ./rq4

### RQ4(a) — Your output

Compare the plot below with the **RQ4(a) reference** shown above (the reference is provided for *this* output).

**What to look for**: same as RQ1(a) — ProToken accuracy should be higher than baselines.

In [ ]:
Image("rq4/scalability_results.png")

### RQ4(b) — Your output

Compare the plot below with the **RQ4(b) reference** shown above (the reference is provided for *this* output).

**What to look for**: same as RQ1(b) — responsible clients’ mean contribution accuracy should be higher than non-responsible clients’.

In [ ]:
Image("rq4/scalability_probability_boxplots.png")